In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier 

In [ ]:
Path = "/Users/ekaterinasorokopudova/Desktop/Tareas repository/Sprint 13/data/df_with_fulltext_embedding.csv"

In [ ]:
df = pd.read_csv(Path)

In [ ]:
def fix_emb(x):
    if isinstance(x, str):
        x = x.strip("[]")
        arr = np.array([float(a) for a in x.split()])
        return arr
    return np.array(x)


In [ ]:
df["embedding"] = df["embedding"].apply(fix_emb)

X = np.vstack(df["embedding"].values)
y = df["queue"]



# Train/test split


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# LightGBM Classifier


In [ ]:

num_classes = y.nunique()

lgbm_clf = LGBMClassifier(
    objective="multiclass",
    num_class=num_classes,
    n_estimators=500,      # число деревьев
    learning_rate=0.05,    # шаг обучения
    max_depth=-1,          # без ограничения глубины, пусть сам подберёт
    random_state=42,
    n_jobs=-1              # использовать все ядра
)

# Обучаем модель
lgbm_clf.fit(X_train, y_train)

# Предсказания на тесте
y_pred = lgbm_clf.predict(X_test)

print("=== LightGBM: classification report ===")
print(classification_report(y_test, y_pred))



# Confusion Matrix (LightGBM)

In [ ]:

plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred, labels=lgbm_clf.classes_)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="GnBu",
    xticklabels=lgbm_clf.classes_,
    yticklabels=lgbm_clf.classes_
)

plt.title("Confusion Matrix: Queue Prediction (LightGBM)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()